# King County House Prices: Refactored Workflow

This is the refactored copy of [King-County.ipynb](King-County.ipynb). The original stays unchanged as a reference.

Data cleaning, feature engineering and modeling now live in the `king_county` package, so this notebook only calls functions and looks at the results:

| Module | Contents |
|---|---|
| `king_county/cleaning.py` | Loading the CSV and the Data Preparation steps |
| `king_county/features.py` | Distance helpers and the Feature Engineering steps |
| `king_county/pipeline.py` | `prepare_data()` for analysis and `make_feature_pipeline()` for models |
| `king_county/model.py` | Train/test split, grid search, evaluation, saving and loading |

The exploratory charts from the original are not repeated here; see the original notebook for the reasoning behind each step.

In [1]:
import sys; print(sys.executable)


c:\Projects\mle-refactoring-project\.venv\Scripts\python.exe


In [ ]:
import plotly.express as px

from king_county.cleaning import clean_data, load_data
from king_county.model import (
    error_by_zipcode,
    error_table,
    evaluate,
    save_model,
    split_data,
    train_model,
)
from king_county.pipeline import FEATURES, prepare_data

## Data Preparation

`clean_data` runs every cleaning step from the original notebook:

1. Drop the implausible 33-bedroom record.
2. Rebuild `sqft_basement` as `sqft_living - sqft_above`, replacing the `?` entries.
3. Fill missing `view` and `waterfront` values with 0, their most common value.
4. Combine `yr_built` and `yr_renovated` into `last_known_change`.

In [3]:
raw = load_data()
clean = clean_data(raw)

print("Raw shape:  ", raw.shape)
print("Clean shape:", clean.shape)
print("Missing values after cleaning:", int(clean.isna().sum().sum()))

Raw shape:   (21597, 21)
Clean shape: (21596, 20)
Missing values after cleaning: 0


## Feature Engineering

`prepare_data` runs cleaning and feature engineering in one call. It adds:

- `sqft_price`: price divided by living plus lot area (for analysis only, because it is derived from the target)
- `center_distance`: km to the "center of wealth" in Medina
- `water_distance`: km to the nearest waterfront house

The water distance is vectorized: the original nested loop took several minutes, this takes well under a second.

In [4]:
kc_data = prepare_data()
kc_data[["price", "sqft_price", "center_distance", "water_distance"]].describe().round(2)

,price,sqft_price,center_distance,water_distance
count,21596.00,21596.00,21596.00,21596.00
mean,540291.96,59.73,17.53,5.54
std,367376.02,45.62,10.15,5.93
min,78000.00,0.16,0.34,0.00
25%,322000.00,28.14,10.33,1.91
50%,450000.00,46.88,14.46,3.42
75%,645000.00,78.02,22.54,6.36
max,7700000.00,399.48,70.19,57.43


## Modeling

`split_data` reproduces the original 70/30 split with `random_state=42`. `train_model` runs the same ElasticNet grid search, but the pipeline starts from raw columns: cleaning and feature engineering run inside each cross-validation fold.

One deliberate difference: `water_distance` now measures only against waterfront houses in the **training** rows. In the original, test houses could use their own waterfront flag, which leaks a little test information into the features.

In [5]:
X_train, X_test, y_train, y_test = split_data(raw)
search = train_model(X_train, y_train)

print("Model features:", len(FEATURES))
print("Best parameters:", search.best_params_)
print(f"Mean cross-validation R^2: {search.best_score_:.3f}")

Model features: 19
Best parameters: {'model__regressor__alpha': 0.01, 'model__regressor__l1_ratio': 0.2}
Mean cross-validation R^2: 0.828


In [6]:
model = search.best_estimator_
metrics = evaluate(model, X_test, y_test)
print(f"Test R^2: {metrics['r2']:.3f}")
print(f"Mean absolute error: ${metrics['mae']:,.0f}")
print(f"Root mean squared error: ${metrics['rmse']:,.0f}")

Test R^2: 0.845
Mean absolute error: $94,808
Root mean squared error: $143,443


**Compared with the original notebook:** the original ElasticNet scored a test R² of **0.844** with `alpha=0.01` and `l1_ratio=0.2`. The refactored model finds the same best parameters and scores **0.845**, so the refactor keeps the same result. The small difference comes from the `water_distance` fix described above. The original only reported R², so MAE and RMSE are new here. Both are in dollars: a typical prediction is off by about $95k, and the larger RMSE shows that a few houses have much bigger errors.

## Error Analysis

`error_table` builds the same residual table as the original notebook. The three largest relative errors are all sales far below the predicted price. The third, house `5111400086`, is the one the original notebook investigated: it sold well below its assessed value.

In [7]:
df_error = error_table(model, X_test, y_test)
df_error.nlargest(3, "price_difference_percent")

,id,latitude,longitude,price,price_prediction,price_difference,price_difference_percent
3344,1437500015,47.7112,-121.315,150000.0,588521.40,438521.40,292.35
2125,5016003230,47.6248,-122.301,169317.0,570412.61,401095.61,236.89
1385,5111400086,47.4235,-122.051,110000.0,334548.86,224548.86,204.14


In [ ]:
# Diverging scale centred on 0: blue = model priced too low, red = too high.
# The symmetric +-50% range keeps white at 0% and stops the few extreme
# houses (up to +292%) from washing out the rest.
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["id", "zipcode", "price", "price_prediction", "price_difference_percent"],
    color="price_difference_percent",
    color_continuous_scale="RdBu_r",
    range_color=[-50, 50],
    opacity=0.7,
    zoom=8.5,
    center={"lat": 47.5, "lon": -122.2},
    height=500,
    labels={"price_difference_percent": "Error %"},
)
fig.update_layout(map_style="open-street-map", margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

### Errors by ZIP code

Individual houses overlap and a few extreme sales stand out, so the map above is better for spotting outliers than for comparing areas. `error_by_zipcode` summarizes the test houses per ZIP code instead:

- **Bubble size**: number of test houses in the ZIP code
- **Colour**: median absolute error in percent, the typical size of a miss there
- **Hover**: mean absolute error in dollars and the bias, where positive means the model overprices that area

In [ ]:
df_zip = error_by_zipcode(df_error)

fig = px.scatter_map(
    df_zip,
    lat="latitude",
    lon="longitude",
    size="houses",
    color="median_abs_percent",
    color_continuous_scale="YlOrRd",
    hover_name="zipcode",
    hover_data={
        "houses": True,
        "median_price": ":$,.0f",
        "mae": ":$,.0f",
        "median_abs_percent": ":.1f",
        "bias_percent": ":+.1f",
        "latitude": False,
        "longitude": False,
    },
    labels={
        "houses": "Test houses",
        "median_price": "Median price",
        "mae": "Mean abs. error",
        "median_abs_percent": "Typical error %",
        "bias_percent": "Bias %",
    },
    size_max=28,
    zoom=8.5,
    center={"lat": 47.5, "lon": -122.2},
    height=500,
)
fig.update_layout(map_style="open-street-map", margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

# The five ZIP codes where the model misses by the most.
df_zip.nlargest(5, "median_abs_percent")

In most ZIP codes the typical miss is 12–19%, but the errors are not random across the map:

- **South Seattle is overpriced.** In 98168, 98108 and 98178 the model's typical prediction is 32–45% above the sale price. These areas are close to the city and the water, so `center_distance` and `water_distance` push their predictions up, but they sold for much less.
- **The most expensive areas are underpriced.** Medina (98039, median price $1.7M) and 98065 come out about 25% too low. Linear terms can't follow how steeply prices climb at the top end.

So the model has a geographic bias that a single R² hides. The model does use `zipcode`, but as a plain number, as if 98168 were "bigger" than 98108. Treating it as a category (one-hot encoding), or adding a neighbourhood price level, would be the next thing to try.

## Save the Model

The saved pipeline includes cleaning and feature engineering, so it predicts directly from raw house records. Load it with `king_county.model.load_model`, which only accepts types from this package.

In [8]:
save_model(model)

WindowsPath('model/model.bin')